In [ ]:
import os
import yaml
import torch
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision.io import decode_image
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(42)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
weights = ResNet50_Weights.DEFAULT # weights = IMAGENET1K_V2
preprocess = weights.transforms()
print(preprocess) # the weights already include the transforms which are needed for the model

resnet_class_names = weights.meta["categories"]  # the class names (list of 1000 strings) are also stored in the weights object
print(resnet_class_names[:20])

#### Current preprocessing
Resize resizes the image to 232 pixels of the smaller edge, then it will be center cropped to 224x224 pixels (the input size of the model) => bilinear interpolation for resizing.
Additionally, the images are normalized with the given mean and std.
This transformation will be applied to all datasets - train, validation and test

In [ ]:
# freeze current model and inspect
model = resnet50(weights=weights) 
for param in model.parameters():
    param.requires_grad = False # freeze the parameters of the model to only train the new layers for object detection first
model

## Test current model for classification on the custom dataset

In [ ]:
# Define the custom dataset and how to load the images and annotations => images and bounding boxes have to transformed equally

class CustomImageDataset(Dataset):
    def __init__(self, annotations_dir, img_dir, img_transform=None):
        self.annotations_dir = annotations_dir
        self.annot_file_names = os.listdir(self.annotations_dir)
        self.img_dir = img_dir
        self.img_transform = img_transform # normal transformations (which will be also applied to validation + test) like resize, crop and normalization

    def __len__(self):
        return len(self.annot_file_names)

    def __getitem__(self, idx):
        file_name = self.annot_file_names[idx]
        # build the path to the annotation.txt and read the content
        annotation_path = os.path.join(self.annotations_dir, file_name)
        with open(annotation_path, "r") as f:
            try:
                objects = f.read().strip().split("\n")
                yolo_boxes = [[float(x.strip()) for x in obj.strip().split(" ")] for obj in objects]  # transform the string to a readable float label array
                # annotations as lines in txt-file with no header, each line contains 5 values seperated by a space: (class_id, x_center, y_center, width, height)
            except Exception as e:
                # no object/label for this image => file is empty, so use an empty label-list as the GT-annotation
                yolo_boxes = []
        # build the related path to the image and load it as a tensor
        img_path = os.path.join(self.img_dir, file_name).removesuffix("txt") + "jpg"
        image = decode_image(img_path)
        # apply preprocess transformations like normalize, which will not affect the label
        if self.img_transform:
            image = self.img_transform(image)
        return image, yolo_boxes

In [ ]:
# Prepare Datasets

dataset_path = "../kaggle/Traffic_Sign/car"

train_dataset =  CustomImageDataset(
    annotations_dir=f"{dataset_path}/train/labels",
    img_dir=f"{dataset_path}/train/images",
    img_transform=preprocess # preprocessing transforms from the weights
)

In [ ]:
# load class names from yaml file

with open(f"{dataset_path}/data.yaml", "r") as f:
    class_names = yaml.safe_load(f)["names"]

print("Num classes:", len(class_names))
print("Classes:", class_names)

In [ ]:
# Prepare Dataloader and define collate function to handle the different amounts of bounding boxes in each image

batch_size = 64
num_workers = 0 # with more num_workers data will be loaded in parallel and the GPU will not have to wait and idle for the CPU to prepare the next batch
# multiple workers will not work in a ipynb on macOS

def collate_fn(batch):
    return list(zip(*batch))
    # with this we prevent an error when returning the batch in the dataloader
    # Pytorch wants to order the batch like this: [(image_1, target_1), (image_2, target_2), ...]
    # here the shape of all images and of all targets must be the same, but with object detection, the shape of the target object varies
    # so we sort new with this by zipping all images and all targets together and turning back into a list:  [(image_1, image_2, ...), (target_1, target_2, ...)] 
    #       => the size of the first and the second tuple are now both 64 


train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_fn)

In [ ]:
# test the current model on some images from the train dataset

model.eval() 
model.to(device)

correct = 0

test_batch = next(iter(train_dataloader))
X, y = torch.stack(test_batch[0]).to(device), test_batch[1]
# batch[0] contains all images in the batch as tuple with len()=64, so all 64 images get stacked to get a tensor with shape [64, 3, 448, 448] 

with torch.no_grad():
    output = model(X)
    pred_idx_batch = output.argmax(dim=1)

pred_class = [resnet_class_names[pred_idx] for pred_idx in pred_idx_batch]
print(pred_class[:10])


In [ ]:
def get_box_coords(box, image_size):
    class_id, x_c, y_c, w, h = box
    width = w * image_size
    height = h * image_size
    x1 = x_c * image_size - width / 2
    y1 = y_c * image_size - height / 2
    return (x1, y1, width, height)

In [ ]:
# for the first few images, show the ground truth bounding box and which classification is predicted by the current network 
# => validation to correct loading, annotation and transformation of the images and bounding boxes

fig, axs = plt.subplots(4, 4, figsize=(15, 12))
axs = axs.flatten()

X = X.cpu() # copy tensor back to cpu
image_size = X.shape[-1]
mean = torch.tensor(preprocess.mean).view(3, 1, 1)
std = torch.tensor(preprocess.std).view(3, 1, 1)

for i in range(16):
   # show the image with the classified label
   # X[i] is the tensor of the normalized image (with mean and std defined above) => with those parameters, we convert back to the normal image
   normal_image = X[i] * std + mean # revert the normalization process by multiplying with the standard deviation and adding the mean
   
   ax = axs[i]
   ax.imshow(normal_image.permute(1, 2, 0))
   ax.set_title(pred_class[i])
   ax.axis("off")

   # draw the boxes around the correct objects
   gt_boxes = [y[i][j] for j in range(len(y[i]))] # get the ground truth boxes for the current image
   for j in range(len(gt_boxes)):
       x1, y1, w, h = get_box_coords(gt_boxes[j], image_size)
       ax.add_patch(plt.Rectangle((x1, y1), w, h, fill=False, edgecolor='red', linewidth=2))

plt.tight_layout()
plt.show()

# some classes in the dataset are not in the ImageNet dataset and the model therefore cannot classify them correctly yet